# Conditional Connection Agreements (CCAs)

This files creates a demonstration of how Conditional Connection Agreemts (CCAs) can be modelled in AggregatorX. CCAs are a flexible grid connection method that enables the DSO to disconnect or limit the grid capacity for the customer under specific conditions such as high demand periods or N-1. 




In [1]:
using HiGHS
using JuMP
using PlotlyJS
using DataFrames

WebIO._IJuliaInit()

In [2]:
# The AggregatorX package needs to be installed before running this:
using AggregatorX

### Example 1:

Example of a charging station, BESS and grid constraint. The charging station will experience some unmet demand

Time-step: 1 hour

In [3]:
relpath = "./"

# Build JuMP modell and solve
opt = HiGHS.Optimizer

# Run the model
f = joinpath(@__DIR__, relpath * "data", "ex_cca1.json")
sys, aggregator = buildaggregator(f);

In [4]:
# See the aggregator object
aggregator


Dict{String, Any} with 9 entries:
  "Connection" => Connection[Connection(1, 2), Connection(2, 3), Connection(3, …
  "Grid"       => Grid[LinearTariff(Dict{Integer, Vector{VariableRef}}(3=>[#und…
  "Group"      => Set(Group[FCReGroup(AffExpr[], AffExpr[], AffExpr[], AffExpr[…
  "Market"     => Market[SimpleDAMarket(Dict{Integer, Vector{VariableRef}}(2=>[…
  "Resource"   => Resource[VariableLoad(Dict{Integer, Vector{VariableRef}}(8=>[…
  "DATADIR"    => "c:\\Users\\astridmo\\python\\megacharge_h3\\data"
  "Node"       => Node[StandardNode(Dict{Integer, AbstractArray{VariableRef}}(5…
  "SYSDIR"     => "c:\\Users\\astridmo\\python\\megacharge_h3"
  "TimeStruct" => IndexedTimeStruct(8)

In [5]:
# Get the components of the aggregator
day_ahead = get_component(1, aggregator);
tpv_constraint = get_component(2, aggregator)
battery = get_component(5, aggregator)
charging_station = get_component(4, aggregator)
fcr = get_component(6, aggregator);
utility_unmet_demand = get_component(8, aggregator);

In [6]:
# Optimize the model
model = optimizeaggregator(aggregator,opt)
is_solved_and_feasible(model)

true

In [7]:
fieldnames(typeof(tpv_constraint))

(:power, :sources, :price, :upper_bound, :id)

In [8]:
daprice = day_ahead.price;
dapower = value.(day_ahead.power[2]);
charging_station_power = value(charging_station.power[8]);
cca_capacity = tpv_constraint.upper_bound;
unmet_demand_price = utility_unmet_demand.price; # EUR/MWh



**Unmet demand:** The unmet demand is defined as the difference between the upper bound of the load (the wanted load), and the power of the load (the delivered power).

**Cost unmet demand:** The cost of unmet demand is defined as the unmet demand * the cost of unmet demand pr MWh. This is defined through the price of the Utility.


In [9]:
# Cost-elements
fcr_income = value(fcr.capacity_sold)' * fcr.price
spot_cost = dapower' * daprice
unmet_demand_mwh = sum(charging_station.upper_bound - charging_station_power)
ratio_covered_demand = (sum(charging_station.upper_bound) - unmet_demand_mwh) / sum(charging_station.upper_bound)
cost_unmet_demand_eur = (charging_station.upper_bound - charging_station_power)' * unmet_demand_price

total_cost = spot_cost - fcr_income + cost_unmet_demand_eur

print("Spot cost: $(round(spot_cost, digits=1))
FCR income: $(round(fcr_income,digits=1)) 
Cost unmet demand: $(round(cost_unmet_demand_eur,digits=1))
**************************************
Total cost: $(round(total_cost,digits=1))\n\n")


Spot cost: 467.3
FCR income: 17.5 
Cost unmet demand: 500.0
**************************************
Total cost: 949.9



Plot the development over time

In [10]:
# N = aggregator["TimeStruct"].periods
# t = [x for x = 1:N]



# battery_soc = value.(battery.state_of_charge); 
# battery_power = value(battery.power[3]);
# charging_station_power = value.(charging_station.power[8]);

# label = ["DA power" "Battery SOC" "Load"]

# p = PlotlyJS.plot(
#     [
#         scatter(x=t, y=dapower,      mode="lines", name="DA power"),
#         scatter(x=t, y=battery_soc,  mode="lines", name="Battery SOC"),
#         scatter(x=t, y=charging_station_power, mode="lines", name="Charging station power"),
#         scatter(x=t, y=cca_capacity, mode="lines", name="Grid capacity", line=attr(color="red", width=3))
#     ],
#     Layout(
#         xaxis_title="Time",
#         yaxis_title="Power (MW)",
#         title="Power of charging station, battery and day-ahead market over time"
#     )
# )

# display(p)

### Example 2: Run several cases to compare the results

The cases can have different:
* Grid capcity (CCA capacity)
* Battery capacity
* Battery power


Time steps: 15 minutes

In [11]:
"""
Run all scenarios for all combinations of (grid capacity limit, battery capacity, battery power) and return a DataFrame
with one row per scenario per month.
"""
function run_scenarios(
    cca_scenarios::AbstractVector{<:Dict{Float64,Float64}},
    cca_original_vector::AbstractVector{<:Real},
    capacities::AbstractVector{<:Real},
    powers::AbstractVector{<:Real};
    run_one
)::DataFrame

    results_df = DataFrame(
        tpv_capacity = Float64[],
        battery_capacity = Float64[],
        battery_power = Float64[],
        spot_cost = Float64[],
        fcr_income = Float64[],
        market_cost = Float64[],
        ratio_covered_demand = Float64[],
        unmet_demand_mwh = Float64[],
        cost_unmet_demand_eur = Float64[],
        total_cost = Float64[],
    )

    for cca_scenario in cca_scenarios, capacity in capacities, power in powers
        # Kjør én scenario-løsning (du gir inn funksjonen)
        r = run_one(cca_scenario, cca_original_vector, float(capacity), float(power))

        # r bør være et NamedTuple med feltene under (uten month/limit/capacity/power)
        push!(results_df, (;
            tpv_capacity = Float64(minimum(tpv_constraint.upper_bound)),
            battery_capacity = Float64(capacity),
            battery_power = Float64(power),
            spot_cost = Float64(r.spot_cost),
            fcr_income = Float64(r.fcr_income),
            market_cost = Float64(r.market_cost),
            ratio_covered_demand = Float64(r.ratio_covered_demand),
            unmet_demand_mwh = Float64(r.unmet_demand_mwh),
            cost_unmet_demand_eur = Float64(r.cost_unmet_demand_eur),
            total_cost = Float64(r.total_cost),
        ))
    end

    return results_df
end


run_one = function(tpv_scenario, cca_original_vector, capacity, power)
    # Change the parameters in the model
    tpv_constraint.upper_bound .= getindex.(Ref(tpv_scenario), cca_original_vector)
    
    battery.capacity = capacity * 4  # Multiply with 4 due to 15 min time steps
    battery.max_charge = power
    battery.max_discharge = power

    # Optimize the model
    model = optimizeaggregator(aggregator,opt)

    ################################################################
    # Hent ut resultater
    daprice = day_ahead.price
    dapower = value.(day_ahead.power[2]);
    battery_soc = value.(battery.state_of_charge) / 4;  # Dividing by 4 to get MWh (from 15 min time resolution)
    battery_power = value(battery.power[3]);
    charging_station_power = value.(charging_station.power[8]);

    ################################################################
    # Calculate results
    fcr_income = (value(fcr.capacity_sold)' * fcr.price) / 4
    spot_cost = (daprice' * dapower) / 4
    market_cost = spot_cost - fcr_income

    total_demand = round(sum(charging_station.upper_bound)/4, digits =3)
    unmet_demand_mwh = round(sum(charging_station.upper_bound - charging_station_power)/4, digits=3)
    ratio_covered_demand = round((sum(charging_station.upper_bound) - sum(charging_station.upper_bound - charging_station_power)) / sum(charging_station.upper_bound), digits=3)
    
    cost_unmet_demand_eur = round(sum(charging_station.upper_bound - charging_station_power)/4 * 1000, digits=0)

    total_cost = market_cost + cost_unmet_demand_eur


    # returnér et NamedTuple med nøyaktig disse nøklene:
    return (
        spot_cost = spot_cost,
        fcr_income = fcr_income,
        market_cost = market_cost,
        ratio_covered_demand = ratio_covered_demand,
        unmet_demand_mwh = unmet_demand_mwh,
        cost_unmet_demand_eur = cost_unmet_demand_eur,
        total_cost = total_cost
    )
end

#3 (generic function with 1 method)

**Daily CCA (D-CCA)**

In [12]:
cca_original_vector = copy(tpv_constraint.upper_bound)

# Changing the grid constraint
tpv_scenarios = [
    Dict(1.0 => 0.001, 2.0 => 2.0),
    Dict(1.0 => 0.5, 2.0 => 2.0),
    Dict(1.0 => 1.0, 2.0 => 2.0),
    Dict(1.0 => 2.0, 2.0 => 2.0),
]  # Changing the TPV constraint

battery_capacities = [2, 1, 0.5, 0.2, 0]  # Change the battery capacity [MWh]
battery_powers = [1, 0.5, 0.2]  # Change the battery power [MW)

results_df = run_scenarios(tpv_scenarios, cca_original_vector, battery_capacities, battery_powers; run_one=run_one)

Row,tpv_capacity,battery_capacity,battery_power,spot_cost,fcr_income,market_cost,ratio_covered_demand,unmet_demand_mwh,cost_unmet_demand_eur,total_cost
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.001,2.0,1.0,110.612,4.9045,105.707,0.889,0.25,250.0,355.707
2,0.001,2.0,0.5,101.828,1.8,100.028,0.811,0.424,424.0,524.028
3,0.001,2.0,0.2,93.578,0.75,92.828,0.745,0.574,574.0,666.828
4,0.001,1.0,1.0,110.612,4.9045,105.707,0.889,0.25,250.0,355.707
5,0.001,1.0,0.5,101.828,1.8,100.028,0.811,0.424,424.0,524.028
6,0.001,1.0,0.2,93.578,0.75,92.828,0.745,0.574,574.0,666.828
7,0.001,0.5,1.0,110.712,3.80625,106.905,0.889,0.25,250.0,356.905
8,0.001,0.5,0.5,101.828,1.8,100.028,0.811,0.424,424.0,524.028
9,0.001,0.5,0.2,93.578,0.75,92.828,0.745,0.574,574.0,666.828


**Seasonal CCA (S-CCA)**

In [13]:
tpv_original_vector = copy(tpv_constraint.upper_bound)

# Changing the grid constraint
tpv_scenarios = [
    Dict(1.0 => 0.001, 2.0 => 0.001),
    Dict(1.0 => 0.5, 2.0 => 0.5),
    Dict(1.0 => 1.0, 2.0 => 1.0),
    Dict(1.0 => 2.0, 2.0 => 2.0),
]  # Changing the TPV constraint

battery_capacities = [2, 1, 0.5, 0.2, 0]  # Change the battery capacity [MWh]
battery_powers = [1, 0.5, 0.2]  # Change the battery power [MW)

results_df = run_scenarios(tpv_scenarios, tpv_original_vector, battery_capacities, battery_powers; run_one=run_one)

Row,tpv_capacity,battery_capacity,battery_power,spot_cost,fcr_income,market_cost,ratio_covered_demand,unmet_demand_mwh,cost_unmet_demand_eur,total_cost
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.001,2.0,1.0,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
2,0.001,2.0,0.5,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
3,0.001,2.0,0.2,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
4,0.001,1.0,1.0,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
5,0.001,1.0,0.5,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
6,0.001,1.0,0.2,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
7,0.001,0.5,1.0,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
8,0.001,0.5,0.5,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04
9,0.001,0.5,0.2,0.1145,0.071,0.0435,0.001,2.248,2248.0,2248.04


In [14]:
function plot_line_total_cost(df, battery_power)

    df_filter = filter(:battery_power => ==(battery_power), df)
    sort!(df_filter, [:battery_capacity, :tpv_capacity])

    traces = [
        scatter(
            x = d.tpv_capacity,
            y = d.total_cost,
            mode = "lines",
            name = "$(bc.battery_capacity) MWh"
        )
        for (bc, d) in pairs(groupby(df_filter, :battery_capacity))
    ]

    plot(
        traces,
        Layout(
            title = attr(
                text = "Total cost for charging station with different grid capacities <br><sup>Battery power = $(battery_power) MW</i></sup>",
                x = 0.5
            ),
            xaxis = attr(title= "Grid capacity limit (MW)", range = [2, -0.8], autorange = "reversed", zeroline = false),
            yaxis_title = "Total cost (EUR) ",
            template = "plotly_white",
            width = 700,
            height = 500,
            margin = attr(l = 80),
            legend = attr(title = attr(text = "Battery capacity"),  x = 0.03, y = 0.90, xanchor = "left", yanchor = "top", bgcolor = "rgba(255,255,255,0.6)", borderwidth = 1), 
            yaxis = attr(tickformat = ",.0f")  # tusenskille
        )
    )
end

plot_line_total_cost (generic function with 1 method)

In [ ]:
plot_line_total_cost(results_df, 1.0)